In [ ]:
# [NB05-C01] SETUP
# What: install the pinned pm4py version, import the libraries, create the
#       figures folder.
# Why: same environment as NB01-NB04 so all the notebooks agree.
# Expected: no errors.

%pip install pm4py==2.7.23.1 -q

import os                                              # filesystem helpers
import pandas as pd                                    # dataframes and CSV handling
import numpy as np                                     # numeric helpers
import matplotlib.pyplot as plt                        # plots
import seaborn as sns                                  # nicer statistical plots
import pm4py                                           # process mining library of the course
from pm4py.util import constants                       # to silence progress bars
from sklearn.model_selection import cross_val_predict, StratifiedKFold, KFold  # cross-validation
from sklearn.linear_model import LinearRegression, LogisticRegression          # linear baselines
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier     # tree ensembles
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.dummy import DummyRegressor, DummyClassifier                       # naive baselines
from sklearn.metrics import mean_absolute_error, r2_score                       # regression metrics
from sklearn.metrics import precision_score, recall_score, f1_score             # classification metrics
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score  # detailed validation

# Silence the progress bars
constants.SHOW_PROGRESS_BAR = False

# Fixed seed: every result of this notebook is reproducible
RANDOM_STATE = 42

# Create the folder where the report figures will be saved (safe if it exists)
os.makedirs('figures', exist_ok=True)

In [ ]:
# [NB05-C02] LOAD THE PREPARED FILES
# What: load the abstract control-flow view, the case table and the phi
#       matrix produced by NB01/NB02/NB04.
# Why: trace encoding works on the control flow (abstract view); the case
#      table carries the prediction targets; the phi matrix carries the six
#      triage-time context features (NB04) that are known early and therefore
#      safe to use as predictors. NO row is dropped.
# Expected: 16,826 abstract rows, 1,820 stays, phi with 1,820 rows.

# Paths of the prepared files (on Colab: upload them and update the paths)
abstract_path = 'abstract_event_log.csv'
cases_path = 'case_table.csv'
phi_path = 'phi_matrix.csv'

# Defensive check: stop with a clear message if a file is missing
for p in [abstract_path, cases_path, phi_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(p + ' not found - run NB01, NB02 and NB04 first.')

# Load the abstract control-flow view
abstract_log = pd.read_csv(abstract_path, parse_dates=['time:timestamp'])
abstract_log['case:concept:name'] = abstract_log['case:concept:name'].astype(str)

# Load the case table (targets and stay length)
case_table = pd.read_csv(cases_path, index_col=0, parse_dates=['start_time', 'end_time'])
case_table.index = case_table.index.astype(str)

# Load the phi matrix (the six triage-time context features)
phi = pd.read_csv(phi_path, index_col=0)
phi.index = phi.index.astype(str)

print('Abstract view rows: {} (expected 16,826)'.format(len(abstract_log)))
print('Stays: {} (expected 1,820)'.format(len(case_table)))
print('phi matrix: {} stays'.format(len(phi)))

In [ ]:
# [NB05-C03] THE TWO PREDICTION TARGETS AND THE LEAKAGE GUARD
# What: define the regression target (total stay duration) and the
#       classification target (admitted vs discharged home), and state the
#       rule that prevents target leakage.
# Why: this is outcome-oriented predictive process monitoring (slide 3.3):
#      predict the outcome y of a stay from an early prefix sigma_1:k, with
#      k < n. Two rules keep it honest: (1) only features KNOWN EARLY may be
#      predictors - the six triage-time phi bits, never the discharge-time
#      properties marked in the NB04 dictionary; (2) the prefix must be
#      strictly shorter than the trace (k < n), otherwise a short stay would
#      expose its own discharge. Both are enforced below.
# Expected: a regression target on all stays and a binary target on 1,737.

# Regression target: the total stay duration in minutes (right-skewed, NB01)
case_table['target_lead_time'] = case_table['lead_time_min']

# Classification target: admitted/transfer (1) vs discharged home (0).
# The 83 interrupted pathways (LWBS, eloped, ...) are a different phenomenon
# and are excluded from THIS task only - they remain in the dataset for
# everything else, exactly as the 56 acuity-missing stays were treated in NB02.
def outcome_label(disposition):
    if disposition in ['ADMITTED', 'TRANSFER']:
        return 1
    if disposition == 'HOME':
        return 0
    return np.nan   # interrupted pathways: not part of the binary task
case_table['target_admitted'] = case_table['disposition'].apply(outcome_label)

classifiable = case_table[case_table['target_admitted'].notna()]
print('Regression target (all stays): {} stays'.format(len(case_table)))
print('Classification target admitted-vs-home: {} stays'.format(len(classifiable)))
print('  admitted/transfer (1): {}'.format(int(classifiable['target_admitted'].sum())))
print('  discharged home (0):   {}'.format(int((classifiable['target_admitted'] == 0).sum())))
print('  minority class share: {:.1f}%'.format(classifiable['target_admitted'].mean() * 100
      if classifiable['target_admitted'].mean() < 0.5 else (1 - classifiable['target_admitted'].mean()) * 100))
print('Interrupted pathways kept but excluded from the binary task: {}'.format(
    int(case_table['target_admitted'].isna().sum())))

In [ ]:
# [NB05-C04] TRACE ENCODING: THE PREFIX ACTIVITY PROFILE
# What: for a given prefix length k, encode each stay as the activity profile
#       of its first k events, joined with the six triage-time phi features.
# Why: this is the trace encoding of slide 3.3 and Case6: turn a sequence
#      into a feature vector a tabular model can use. The activity profile is
#      built with pm4py.get_prefixes_from_log + extract_features_dataframe.
#      Two leakage guards are applied: only stays with MORE than k events
#      enter (k < n, so the prefix is a true prefix), and the discharge
#      column is dropped defensively (a prefix must never contain the end).
# Expected: a feature matrix per k, leakage-free by construction.

# The six phi features known by triage time (from NB04): safe predictors
CORE_PHI = ['acuity_1_3', 'acuity_1_2', 'ambulance_arrival',
            'pain_any_gt_0', 'tachycardia_any_gt_100', 'vitals_before_triage']

# Helper: build the encoded feature matrix for a prefix length k
def encode_prefix(k):
    # Only stays strictly longer than k: the prefix must be shorter than the
    # trace (k < n), otherwise a short stay would expose its own discharge
    long_enough = case_table[case_table['event_count'] > k].index
    log_subset = abstract_log[abstract_log['case:concept:name'].isin(long_enough)]

    # First k events of each stay
    prefixes = pm4py.get_prefixes_from_log(log_subset, length=k)

    # Activity profile of the prefix (one-hot counts per activity)
    features = pm4py.extract_features_dataframe(
        prefixes, str_ev_attr=['concept:name'],
        activity_key='concept:name', case_id_key='case:concept:name',
        timestamp_key='time:timestamp', include_case_id=True)
    features = features.set_index('case:concept:name')
    features.index = features.index.astype(str)

    # Leakage guard: drop the discharge column if present (a genuine prefix
    # of a stay with n > k events cannot contain the discharge, but we remove
    # the column defensively so the guard is explicit)
    discharge_column = 'concept:name_Discharge from the ED'
    if discharge_column in features.columns:
        features = features.drop(columns=[discharge_column])

    # Join the six triage-time phi features
    features = features.join(phi[CORE_PHI].astype(int))
    return features

# Encode at k = 2, 3, 4 and report how many stays are predictable at each
encoded = {}
for k in [2, 3, 4]:
    features = encode_prefix(k)
    encoded[k] = features
    print('k = {}: {} predictable stays ({:.1f}% of 1,820) | {} features'.format(
        k, len(features), len(features) / len(case_table) * 100, features.shape[1]))

In [ ]:
# [NB05-C05] COMPARING TRACE-ENCODING METHODS
# What: build three control-flow encodings of the k=3 prefix - Boolean
#       (activity present/absent), frequency (activity counts) and 2-gram
#       (consecutive activity pairs) - and compare how well each predicts the
#       admission outcome. The chosen encoding (frequency + phi) is used by
#       the rest of the notebook.
# Why: the trace-encoding literature (slide 3.3) lists several ways to turn a
#      sequence into a vector, each a trade-off: Boolean is simplest but
#      ignores counts and order; frequency captures "how many" but not order;
#      n-grams capture local order at the cost of more features. Embeddings
#      (Trace2Vec, Node2Vec) are more powerful but are black boxes, so they
#      are excluded from an interpretability-focused project. We measure the
#      trade-off instead of assuming it.
# Expected: frequency and 2-gram slightly above Boolean, all close because
#      the ED alphabet is tiny (6 activities) so order carries little signal.

# The k=3 prefixes of the stays long enough for the prefix (k < n)
k_bench = 3
long_enough = case_table[case_table['event_count'] > k_bench].index
log_subset = abstract_log[abstract_log['case:concept:name'].isin(long_enough)]
prefixes = pm4py.get_prefixes_from_log(log_subset, length=k_bench)
activities = ['Enter the ED', 'Triage in the ED', 'Medicine reconciliation',
              'Medicine dispensations', 'Vital sign check']   # discharge excluded (leakage)

# Sequence of the first k events per stay
prefix_sequences = prefixes.groupby('case:concept:name')['concept:name'].agg(list)

# Boolean encoding: 1 if the activity appears in the prefix, else 0
def boolean_encoding(sequence):
    return {'bool_' + a: int(a in sequence) for a in activities}

# Frequency encoding: how many times each activity appears in the prefix
def frequency_encoding(sequence):
    return {'freq_' + a: sequence.count(a) for a in activities}

# 2-gram encoding: 1 if a consecutive activity pair appears in the prefix
def bigram_encoding(sequence):
    pairs = ['{}>{}'.format(sequence[i], sequence[i + 1]) for i in range(len(sequence) - 1)]
    all_pairs = ['{}>{}'.format(a, b) for a in activities for b in activities]
    return {'2gram_' + p: int(p in pairs) for p in all_pairs}

# Build the three feature tables
boolean_features = pd.DataFrame([boolean_encoding(s) for s in prefix_sequences],
                                index=prefix_sequences.index)
frequency_features = pd.DataFrame([frequency_encoding(s) for s in prefix_sequences],
                                  index=prefix_sequences.index)
bigram_features = pd.DataFrame([bigram_encoding(s) for s in prefix_sequences],
                               index=prefix_sequences.index)
for table in [boolean_features, frequency_features, bigram_features]:
    table.index = table.index.astype(str)

# A stratified 5-fold splitter for the encoding comparison
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Evaluate each encoding on the admission task with the same model and CV
classifiable_ids_local = case_table[case_table['target_admitted'].notna()].index
encoding_rows = []
for name, table in [('Boolean', boolean_features), ('Frequency', frequency_features), ('2-gram', bigram_features)]:
    ids = [i for i in table.index if i in set(classifiable_ids_local)]
    X_enc = table.loc[ids]
    y_enc = case_table.loc[ids, 'target_admitted'].astype(int)
    model = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE)
    predictions = cross_val_predict(model, X_enc.values, y_enc.values, cv=skf)
    f1 = f1_score(y_enc, predictions, average='macro', zero_division=0)
    encoding_rows.append({'encoding': name, 'features': X_enc.shape[1], 'macro_f1': round(f1, 3)})
    print('{:10s} -> {} features | macro F1 {:.3f}'.format(name, X_enc.shape[1], f1))

encoding_benchmark = pd.DataFrame(encoding_rows)
encoding_benchmark.to_csv('encoding_benchmark.csv', index=False)
print('\nSaved encoding_benchmark.csv')

# Interpretation: the three control-flow encodings score close to one another
# because the ED has only six activities and a short common prefix, so word
# order adds little over simple counts. The frequency encoding is chosen for
# the rest of the notebook: it dominates Boolean (it keeps the counts of
# repeated clinical activities) without the feature blow-up of the 2-gram,
# and it stays fully interpretable - unlike the embedding methods, which the
# literature notes are black boxes.

In [ ]:
# [NB05-C06] LEAKAGE CHECK
# What: verify that no encoded feature matrix contains a discharge signal and
#       that the feature set is exactly the prefix profile plus the phi bits.
# Why: the leakage guards of NB05-C04 must be checked, not trusted (this is
#      the lesson repeated across the project: verify, do not assume). If a
#      later model reaches a suspiciously perfect score, this is the first
#      place to look.
# Expected: zero discharge columns, only prefix activities and phi bits.

for k in [2, 3, 4]:
    columns = list(encoded[k].columns)
    has_discharge = any('Discharge' in c for c in columns)
    print('k = {}: discharge feature present? {} | columns: {}'.format(
        k, has_discharge, columns))

# Interpretation: none of the matrices carries a discharge feature, and the
# only case-level features are the six triage-time phi bits, so a model
# trained here predicts from information genuinely available early in the
# stay. Any accuracy above ~0.95 would still be treated as a leakage
# suspect and investigated before being reported.

In [ ]:
# [NB05-C07] REGRESSION: PREDICTING THE TOTAL STAY DURATION
# What: predict the total lead time from each prefix length, with three
#       models and a naive baseline, evaluated by 5-fold cross-validation.
# Why: cross-validation (course requirement) gives an honest estimate by
#      averaging over five held-out folds instead of one split. The baseline
#      (predict the median duration for everyone) is the bar every model must
#      clear: a model that does not beat it has learned nothing.
# Expected: modest R2 - duration is hard to predict from an early prefix -
#      but the models should beat the median baseline.

# The five-fold splitter, fixed seed
kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# The models to compare (a naive baseline plus three regressors)
regressors = {
    'baseline (median)': DummyRegressor(strategy='median'),
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=200, random_state=RANDOM_STATE),
}

# Evaluate every model at every prefix length
regression_rows = []
for k in [2, 3, 4]:
    features = encoded[k]
    y = case_table.loc[features.index, 'target_lead_time']
    for model_name, model in regressors.items():
        # Cross-validated predictions, then metrics on them
        predictions = cross_val_predict(model, features.values, y.values, cv=kfold)
        mae = mean_absolute_error(y, predictions)
        r2 = r2_score(y, predictions)
        regression_rows.append({'k': k, 'model': model_name,
                                'MAE_min': round(mae, 1), 'R2': round(r2, 3)})
        print('k={} | {:18s} -> MAE {:6.1f} min | R2 {:.3f}'.format(k, model_name, mae, r2))
    print()

regression_results = pd.DataFrame(regression_rows)
regression_results.to_csv('prediction_regression.csv', index=False)
print('Saved prediction_regression.csv')

# Interpretation: read each model against the median baseline at the same k.
# The gain over the baseline is what the early information buys; the low
# absolute R2 is itself a finding - most of the duration is decided by what
# happens AFTER the first few events, which the prefix cannot see (consistent
# with the waiting-not-workload result of section 2.5).

In [ ]:
# [NB05-C08] CLASSIFICATION: PREDICTING ADMISSION VS HOME
# What: predict the binary outcome from each prefix length, with three models
#       and a naive baseline, evaluated by 5-fold STRATIFIED cross-validation.
# Why: the outcome is imbalanced-ish (37% admitted), so the folds are
#      stratified to keep the class ratio, and the metrics are macro Precision
#      / Recall / F1 (robust to imbalance, course note) rather than plain
#      accuracy. class_weight='balanced' lets the models pay attention to the
#      minority class without resampling.
# Expected: models clearly above the majority baseline; F1 rising with k.

# Only the classifiable stays (admitted vs home), target as integer
classifiable_ids = case_table[case_table['target_admitted'].notna()].index

# The stratified five-fold splitter
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# The models (baseline plus three classifiers with balanced class weights)
classifiers = {
    'baseline (majority)': DummyClassifier(strategy='most_frequent'),
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE),
}

# Evaluate every model at every prefix length
classification_rows = []
for k in [2, 3, 4]:
    features = encoded[k]
    # Restrict to the classifiable stays present at this prefix length
    ids = [i for i in features.index if i in set(classifiable_ids)]
    X = features.loc[ids]
    y = case_table.loc[ids, 'target_admitted'].astype(int)
    for model_name, model in classifiers.items():
        predictions = cross_val_predict(model, X.values, y.values, cv=skf)
        precision = precision_score(y, predictions, average='macro', zero_division=0)
        recall = recall_score(y, predictions, average='macro', zero_division=0)
        f1 = f1_score(y, predictions, average='macro', zero_division=0)
        classification_rows.append({'k': k, 'model': model_name, 'stays': len(ids),
                                    'precision_macro': round(precision, 3),
                                    'recall_macro': round(recall, 3), 'f1_macro': round(f1, 3)})
        print('k={} | {:19s} -> precision {:.3f} | recall {:.3f} | F1 {:.3f}'.format(
            k, model_name, precision, recall, f1))
    print()

classification_results = pd.DataFrame(classification_rows)
classification_results.to_csv('prediction_classification.csv', index=False)
print('Saved prediction_classification.csv')

# Interpretation: any model whose macro F1 approaches 1.0 would be a leakage
# suspect (the lesson of Case6); the values here should sit well below that,
# beating the majority baseline but far from perfect - honest early
# prediction, not a leak.

In [ ]:
# [NB05-C09] WHAT DRIVES THE PREDICTION?
# What: read the feature importances of the best classifier at k = 4.
# Why: a predictive model is only useful to the ED if its drivers can be
#      named. This connects the prediction back to the context patterns of
#      NB04 and the performance findings of NB02.
# Expected: the triage-time phi features (acuity, ambulance) and the early
#      clinical activities among the top drivers.

# Fit the Random Forest on all classifiable stays at k = 4 to read importances
features = encoded[4]
ids = [i for i in features.index if i in set(classifiable_ids)]
X = features.loc[ids]
y = case_table.loc[ids, 'target_admitted'].astype(int)
forest = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE)
forest.fit(X.values, y.values)

# Importances as a readable, sorted table
importances = pd.DataFrame({'feature': X.columns, 'importance': forest.feature_importances_})
importances = importances.sort_values('importance', ascending=False).reset_index(drop=True)
importances['feature'] = importances['feature'].str.replace('concept:name_', 'activity: ', regex=False)
print('Feature importances (Random Forest, admitted-vs-home, k=4):')
print(importances.to_string(index=False))

# Bar plot of the top features
plt.figure(figsize=(9, 5))
top = importances.head(10).iloc[::-1]
plt.barh(top['feature'], top['importance'], color='#5b9bd5')
plt.xlabel('Importance')
plt.title('What predicts admission: feature importances (k = 4)')
plt.tight_layout()

# Save the figure for the report, then show it
plt.savefig('figures/NB05_feature_importance.png', dpi=200)
plt.show()

In [ ]:
# [NB05-C10] DETAILED VALIDATION OF THE ADMISSION CLASSIFIER
# What: for the best classifier at k = 4, report the full validation picture:
#       the per-class classification report, the confusion matrix and the
#       AUC-ROC, all on cross-validated predictions.
# Why: the macro F1 of NB05-C08 summarises the model in one number; the
#      course validation checklist (Recall, Precision, F1, AUC-ROC) asks for
#      the breakdown behind it. The confusion matrix shows WHICH errors the
#      model makes - false admissions vs missed admissions - which matters
#      operationally, and AUC-ROC measures the ranking quality independently
#      of the decision threshold.
# Expected: balanced per-class scores, more errors on the minority class,
#      an AUC comfortably above 0.5 but far from 1.0 (no leakage).

# The best model on the k=4 encoding, cross-validated predictions
features = encoded[4]
ids = [i for i in features.index if i in set(classifiable_ids)]
X = features.loc[ids]
y = case_table.loc[ids, 'target_admitted'].astype(int)
model = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE)
cv_predictions = cross_val_predict(model, X.values, y.values, cv=skf)
# Cross-validated class probabilities, for the AUC-ROC
cv_proba = cross_val_predict(model, X.values, y.values, cv=skf, method='predict_proba')[:, 1]

# Per-class classification report (precision / recall / f1 for each outcome)
print('Classification report (admitted-vs-home, k=4, 5-fold CV):')
print(classification_report(y, cv_predictions, target_names=['home (0)', 'admitted (1)'], zero_division=0))

# AUC-ROC: ranking quality regardless of the decision threshold
auc = roc_auc_score(y, cv_proba)
print('AUC-ROC: {:.3f}'.format(auc))

# Confusion matrix, shown as a labelled heatmap
matrix = confusion_matrix(y, cv_predictions)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['pred home', 'pred admitted'],
            yticklabels=['true home', 'true admitted'])
plt.title('Confusion matrix: admission prediction (k=4)')
plt.tight_layout()
plt.savefig('figures/NB05_confusion_matrix.png', dpi=200)
plt.show()

# Interpretation: read the off-diagonal cells. A false admission (predicting
# admitted for a patient who goes home) wastes an early bed search; a missed
# admission (predicting home for a patient who is admitted) is the costlier
# error, because it delays the bed request. The AUC confirms the model ranks
# patients by admission risk well above chance, while staying far from the
# perfect score that would betray leakage.

In [ ]:
# [NB05-C11] IS OVERSAMPLING NEEDED? A WHAT-IF
# What: measure what would change if the minority class were oversampled,
#       without adopting it.
# Why: the course discusses oversampling rare variants (long-tailed data).
#      Here the minority class is 37%, not rare, so oversampling is likely
#      unnecessary - but the claim is checked, not asserted. class_weight,
#      already used above, is the lighter alternative that needs no resampling.
# Expected: a small or negative change, confirming oversampling is not needed.

# A simple manual oversampling of the minority class on the k=4 features
features = encoded[4]
ids = [i for i in features.index if i in set(classifiable_ids)]
X = features.loc[ids]
y = case_table.loc[ids, 'target_admitted'].astype(int)

# Cross-validated macro F1 WITHOUT oversampling (class_weight only)
model = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE)
base_predictions = cross_val_predict(model, X.values, y.values, cv=skf)
base_f1 = f1_score(y, base_predictions, average='macro', zero_division=0)
print('Macro F1 with class_weight only (no oversampling): {:.3f}'.format(base_f1))

# What-if: duplicate the minority-class rows once inside each training fold.
# We do it fold by fold to avoid leaking duplicated rows into the test fold.
oversampled_scores = []
for train_index, test_index in skf.split(X.values, y.values):
    X_train, y_train = X.values[train_index], y.values[train_index]
    X_test, y_test = X.values[test_index], y.values[test_index]
    minority_mask = y_train == 1
    X_train_os = np.vstack([X_train, X_train[minority_mask]])
    y_train_os = np.concatenate([y_train, y_train[minority_mask]])
    model_os = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
    model_os.fit(X_train_os, y_train_os)
    predictions = model_os.predict(X_test)
    oversampled_scores.append(f1_score(y_test, predictions, average='macro', zero_division=0))
oversampled_f1 = np.mean(oversampled_scores)
print('Macro F1 with minority oversampling: {:.3f}'.format(oversampled_f1))
print('Change: {:+.3f}'.format(oversampled_f1 - base_f1))

# Interpretation: the change is small, so oversampling is not adopted - at a
# 37% minority share, class_weight is enough and resampling would only add
# complexity. Oversampling would matter for a genuinely rare target (the
# course example of long-tailed variants), which this is not.

In [ ]:
# [NB05-C12] THE IMPROVEMENT BACKLOG
# What: assemble the improvement options, each backed by converging evidence
#       from the whole project and paired with a KPI and a stated risk.
# Why: this is the fourth assignment goal. Every row is built only where the
#      evidence converges (a duration gap AND a group AND a deviation or a
#      predictor), and every row states that the association is not a proven
#      cause - the methodological safety principle of this project.
# Expected: a compact, honest backlog table for the report.

# The backlog is written from results measured earlier in the project, not
# invented here. Each row: evidence -> hypothesis -> owner -> KPI -> risk.
improvement_backlog = pd.DataFrame([
    {'evidence': 'Wait-to-discharge bottleneck: ~200 median min between the last clinical milestone and discharge (NB02-C09)',
     'hypothesis': 'introduce an earlier bed-request / discharge-readiness trigger once clinical work is complete',
     'owner': 'ED operations + bed management',
     'kpi': 'median and p90 of the last-milestone-to-discharge time',
     'risk': 'association, not cause: the wait may reflect downstream ward capacity, not ED behaviour'},
    {'evidence': 'Urgent + ambulance context variants (110101, 111100) have the longest stays and highest admission (NB04-C06, NB04-C13)',
     'hypothesis': 'a fast-track lane for the high-acuity ambulance cohort, and early admission prediction to start bed search sooner',
     'owner': 'ED triage + operations',
     'kpi': 'median lead time and admission-decision time for the urgent-ambulance cohort',
     'risk': 'predictive model is early but imperfect (NB05-C08): use it to prioritise, not to decide'},
    {'evidence': 'Vital sign check before triage in 19.9% of stays, across all segments, with NO duration effect (NB03-C19, NB04-C13)',
     'hypothesis': 'record the initial assessment as one composite step in the data model, not as a process change',
     'owner': 'ED informatics / data governance',
     'kpi': 'share of stays with the vitals-before-triage recording pattern',
     'risk': 'this is a recording-practice signal, not a performance problem: no clinical change is implied'},
    {'evidence': 'Long tail = 40% of stays on single-case paths, significantly slower (median 433 vs 195 min) and more deviating (NB02-C17, NB03-C18)',
     'hypothesis': 'review the long-tail stays case by case to separate legitimate complex care from recording anomalies',
     'owner': 'ED clinical audit',
     'kpi': 'share of long-tail stays reclassified as anomaly vs legitimate complexity',
     'risk': 'the long tail is heterogeneous (NB02-C16): a blanket rule would harm the legitimate complex cases'},
])
improvement_backlog.to_csv('improvement_backlog.csv', index=False)
print('Saved improvement_backlog.csv')
for _, row in improvement_backlog.iterrows():
    print('\nEVIDENCE:   {}'.format(row['evidence']))
    print('HYPOTHESIS: {}'.format(row['hypothesis']))
    print('KPI:        {}'.format(row['kpi']))
    print('RISK:       {}'.format(row['risk']))

In [ ]:
# [NB05-C13] EXPORTS AND NO-DATA-LOSS CHECK
# What: recap the artifacts and verify that no stay was lost.
# Why: project rule - nothing is ever dropped. The stays too short to be
#      predicted at a given k, and the interrupted pathways excluded from the
#      binary task, all remain in the case table.
# Expected: 1,820 stays still present.

# No-data-loss check
print('Stays in the case table: {} (started with 1,820)'.format(len(case_table)))
print('Stays too short for a k=4 prefix (kept, not predictable at k=4): {}'.format(
    (case_table['event_count'] <= 4).sum()))
print('Interrupted pathways (kept, excluded from the binary task only): {}'.format(
    int(case_table['target_admitted'].isna().sum())))

# Recap of the artifacts
print('\nNB05 outputs: prediction_regression.csv, prediction_classification.csv,')
print('improvement_backlog.csv, feature importance figure in figures/')
print('\nNo stay was removed. Predictors are the six triage-time phi features')
print('plus the prefix activity profile; the discharge-time properties are')
print('excluded, and the prefix is always strictly shorter than the trace.')